# **Radiotherapy Experiment**

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

### ***Input the location of the file to be analysed from the DoseProfileMC simulation***

In [ ]:
fileName = '/home/simong/data/DoseMC_ion_3075MeV_1000n_Tumour_Out.csv'

**Parse the input file header which describes the voxelisation of the simulated phantom. The units of the phantom dimensions are in cm**

In [ ]:
header = pd.read_csv(fileName,nrows=1)

xbins  = header["xbins"][0]
xmin   = header["xmin"][0]
xmax   = header["xmax"][0]

ybins  = header["ybins"][0]
ymin   = header["ymin"][0]
ymax   = header["ymax"][0]

zbins  = header["zbins"][0]
zmin   = header["zmin"][0]
zmax   = header["zmax"][0]

intnev   = header["nevent"][0]
particle = header["particle"][0]
energy   = header["energy"][0]

header

**Parse the input file and fill a dataframe, converting numbered voxels into positions.**

In [ ]:
dataframe = pd.read_csv(fileName,skiprows=2)
dataframe["sum"]     = dataframe["lepton"]+dataframe["meson"]+dataframe["baryon"]+dataframe["ion"]
dataframe["sumT"]    = dataframe["leptonT"]+dataframe["mesonT"]+dataframe["baryonT"]+dataframe["ionT"]
dataframe["leptonA"] = dataframe["lepton"]+dataframe["leptonT"]
dataframe["mesonA"]  = dataframe["meson"]+dataframe["mesonT"]
dataframe["baryonA"] = dataframe["baryon"]+dataframe["baryonT"]
dataframe["ionA"]    = dataframe["ion"]+dataframe["ionT"]
dataframe["Total"]   = dataframe["sum"]+dataframe["sumT"]
dataframe["xpos"]    = xmin+(0.5+dataframe["x"])*(xmax-xmin)/xbins
dataframe["ypos"]    = ymin+(0.5+dataframe["y"])*(ymax-ymin)/ybins
dataframe["zpos"]    = zmin+(0.5+dataframe["z"])*(zmax-zmin)/zbins
dataframe

**Create a dataframe to store the simulated data. Remove non-zero values.**

In [ ]:
filterframe = dataframe[(dataframe['Total']>0)]

***
**Example plotting code is given below to perform the analysis. Feel free to modify and add to the below**
***

**Dose by particle type:**

In [ ]:
#plot dose by particle type
nbins = 100
erange = 0,filterframe["Total"].max()

FigureE = plt.figure(figsize=(15,10))
histL = filterframe['leptonA'].hist(bins=nbins, range=erange, color='black',   alpha=0.5, label='Leptons')
histM = filterframe['mesonA'] .hist(bins=nbins, range=erange, color='red',     alpha=0.5, label="Mesons" )
histB = filterframe['baryonA'].hist(bins=nbins, range=erange, color='blue',    alpha=0.5, label="Baryons")
histI = filterframe['ionA']   .hist(bins=nbins, range=erange, color='magenta', alpha=0.5, label="Ions"   )
plt.xlabel('Energy Deposited (MeV)')
leg=plt.legend()
plt.yscale('log')
plt.show()

#plt.savefig('DoseByParticle.pdf')  

In [ ]:
# calculate energy deposited by each particle type
print('For',intnev,'incident',particle,'with an energy of',energy, 'MeV')
# Sum of leptons, mesons, baryons, ions
CalcTotal = filterframe['Total'].sum()
print('Calculated total energy deposited by leptons, mesons, baryons and ions ', CalcTotal, 'MeV')

# leptons
Leptons = filterframe['leptonA'].sum()
LeptonFrac = Leptons/CalcTotal
print('Fraction of energy deposited by leptons = ', LeptonFrac*100, " %")

# mesons
Mesons = filterframe['mesonA'].sum()
MesonFrac = Mesons/CalcTotal
print('Fraction of energy deposited by mesons = ', MesonFrac*100, " %")

# baryons
Baryons = filterframe['baryonA'].sum()
BaryonFrac = Baryons/CalcTotal
print('Fraction of energy deposited by baryons = ', BaryonFrac*100, " %")

# ions
Ions = filterframe['ionA'].sum()
IonFrac = Ions/CalcTotal
print('Fraction of energy deposited by ions = ', IonFrac*100, " %")


**Longitudinal Dose Profile:**

In [ ]:
rangey = ymin,ymax
rangex = xmin,xmax
rangez = zmin,zmax

cutframeL = filterframe[(abs(filterframe['zpos'])<10) & (abs(filterframe['xpos'])<10)]

FigureL = plt.figure(figsize=(15,10))
plt.hist(cutframeL['ypos'],weights=cutframeL['sum']/intnev,bins=ybins,range=rangey, alpha=0.5, label="Normal")
plt.hist(cutframeL['ypos'],weights=cutframeL['sumT']/intnev,bins=ybins,range=rangey, alpha=0.5, label="Tumour")
plt.xlabel('Longitudinal Position (mm)')
leg=plt.legend()
plt.show()

# calculate the mean voxel energy deposit per incident particle
TotalSumL = filterframe['sum'].sum() + filterframe['sumT'].sum()
MeanVoxelEL = TotalSumL/(intnev*ybins)
print('Mean energy deposited per voxel in longitudinal direction ',MeanVoxelEL, 'MeV per voxel')

# calculate TNR
TNRl = filterframe['sumT'].sum() / filterframe['sum'].sum()
print('TNR is ', TNRl)

#plt.savefig('LongitudalProfile.pdf')  



**Transverse Dose Profile:**



In [ ]:
cutframeT = filterframe[abs(filterframe['ypos'])<10]

FigureT = plt.figure(figsize=(15,10))
plt.hist(cutframeT['xpos'],weights=cutframeT['sum']/intnev, bins=xbins,range=rangex, alpha=0.5, label="Normal")
plt.hist(cutframeT['xpos'],weights=cutframeT['sumT']/intnev,bins=xbins,range=rangex, alpha=0.5, label="Tumour")
plt.xlabel('Transverse Position (mm)')
leg=plt.legend()
plt.show()

# calculate the mean voxel energy deposit per incident particle
TotalSumL = cutframeT['sum'].sum() + cutframeT['sumT'].sum()

MeanVoxelET = TotalSumL/(intnev*zbins)
print('Mean energy deposited per voxel in transverse direction ',MeanVoxelET, 'MeV per voxel')

#plt.savefig('TransverseProfile.pdf')  

In [ ]:
#2-dimensional transverse energy deposition for visualisation purposes

rangexz = [xmin,xmax],[zmin,zmax]

#first energy in healthy tissue
FigureXZ = plt.figure(figsize=(20,15))
plt.hist2d(x=filterframe['xpos'], y=filterframe['zpos'], weights=filterframe['Total'], bins=(xbins,zbins), range=rangexz, cmap=plt.cm.jet)
cbar=plt.colorbar()
plt.xlabel('X Position (mm)')
plt.ylabel('Z Position (mm)')
cbar.ax.set_ylabel('Energy deposited (MeV)')
plt.show()

#plt.savefig('TransverseProfile2D.pdf')  